# Customer Segmentation for E-Commerce — End-to-End Analysis

**Author:** _Your Name_
**Project type:** Unsupervised Machine Learning (Clustering) — Internship Submission

---

### The business problem

Imagine you're the newly hired data analyst at a growing online retail company. The marketing
team has one budget but thousands of very different customers — students who hunt for
discounts, busy professionals who buy premium items without blinking, and loyal shoppers who
haven't ordered in months and might be slipping away.

Right now the company sends **the same email, the same discount code, and the same ad** to
everyone. That's expensive, and it's not working as well as it should.

**Our job:** group customers into a small number of meaningful segments based on their real
behavior (spending, frequency, recency, income) so marketing can design a *different, sharper
strategy for each group* instead of one blurry strategy for everyone.

### Data note (real + engineered)
This project is built on the **real "Mall Customer Segmentation Data" from Kaggle**
(`vjchoudhary7/customer-segmentation-tutorial-in-python`) — 200 genuine customer records with
`Gender`, `Age`, `Annual Income`, and `Spending Score`. That real dataset is intentionally small
and doesn't include transaction-level fields like recency or purchase history, so it can't
support a full RFM analysis on its own. To meet the scope of this project we **expanded** the
200 real rows to 1,200 (via resampling + small realistic jitter that preserves the original
distribution) and **added behavioral columns** (Recency, Purchase Frequency, Tenure, Category,
City, Monetary Value) that are simulated *conditionally* on each customer's real Spending
Score/Income — not random noise. See `src/build_dataset.py` for the fully documented, reproducible
logic, and the README for a clear "real vs. engineered" column list.

### What this notebook covers
1. Load & inspect the data
2. Exploratory Data Analysis (EDA) — distributions, outliers, correlations
3. Feature engineering — RFM analysis, encoding, scaling
4. Clustering — Elbow method, Silhouette score, K-Means vs Hierarchical vs DBSCAN
5. Dimensionality reduction (PCA) & visualization
6. Cluster profiling, naming, and business recommendations
7. Model evaluation & stability check


## 1. Setup

In [ ]:
import sys, warnings
sys.path.append("../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import (
    load_data, handle_missing_values, cap_outliers_iqr,
    build_rfm, encode_categoricals, scale_features
)
from clustering import (
    find_optimal_k, run_kmeans, run_hierarchical, run_dbscan,
    evaluate_clustering, reduce_dimensions, cluster_stability
)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42


## 2. Load & Inspect the Data

We start with the merged dataset (`data/customer_data.csv`) — real Kaggle base + engineered behavioral columns.

In [ ]:
df = load_data("../data/customer_data.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

**Insight:** We have 1,200 customers and 13 columns. `Age`, `AnnualIncome_INR_K`,
`SpendingScore`, `PurchaseFrequency`, `Recency_Days`, `Tenure_Months` are our key numeric
behavioral signals. `Gender`, `City`, `ProductCategoryPreference` are categorical.

## 3. Missing Values & Outliers

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0]
print(missing)

plt.figure(figsize=(6,4))
sns.barplot(x=missing.values, y=missing.index, color="#C44E52")
plt.title("Missing Values by Column")
plt.xlabel("Count of missing values")
plt.tight_layout()
plt.show()

**Insight:** A small, realistic amount of missing data (~2%) exists in `City`,
`AnnualIncome_INR_K`, and `SatisfactionScore` — mirroring gaps you'd see in real CRM exports.
We impute numeric columns with the median and categorical columns with the mode, since ~2%
missingness is too small to justify dropping rows and won't meaningfully distort the
distribution.

In [ ]:
df = handle_missing_values(df)
print("Remaining missing values:", df.isna().sum().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ["AnnualIncome_INR_K", "SpendingScore", "MonetaryValue_INR"]):
    sns.boxplot(y=df[col], ax=ax, color="#DD8452")
    ax.set_title(f"Boxplot: {col}")
plt.tight_layout()
plt.show()

**Insight:** `AnnualIncome_INR_K` has a handful of extreme high-income outliers
(the "whale" customers). Rather than deleting them — they're valuable premium customers, not
data errors — we **cap them at the 1.5×IQR fence** (winsorization). This keeps them in the
dataset but prevents them from dominating distance-based clustering later.

In [ ]:
df = cap_outliers_iqr(df, ["AnnualIncome_INR_K", "MonetaryValue_INR", "AvgOrderValue_INR"])

## 4. Univariate Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
num_cols = ["Age", "AnnualIncome_INR_K", "SpendingScore", "PurchaseFrequency", "Recency_Days", "Tenure_Months"]
for ax, col in zip(axes.flat, num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="#4C72B0")
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

**Insight:**
- `Age` is roughly centered in the 25-55 range — a broad adult customer base, not skewed to one generation.
- `SpendingScore` is fairly spread out, hinting that spending behavior will be a strong differentiator for clustering.
- `Recency_Days` is right-skewed — most customers bought recently, but there's a long tail of customers who haven't ordered in a long time (early warning sign of churn risk).

## 5. Bivariate Analysis

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(data=df, x="AnnualIncome_INR_K", y="SpendingScore", hue="Gender", alpha=0.6)
plt.title("Annual Income vs Spending Score")
plt.tight_layout()
plt.show()

**Insight:** Income alone does NOT predict spending score — there are high-income
customers who spend little and low-income customers who spend a lot. This is exactly why we
need **multi-dimensional clustering** rather than simply segmenting by income.

In [ ]:
plt.figure(figsize=(10, 8))
corr = df[num_cols + ["MonetaryValue_INR", "SatisfactionScore"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

**Insight:** `SpendingScore` and `PurchaseFrequency` correlate positively with
`MonetaryValue_INR` (as expected — customers who buy more often and rate higher spending scores
generate more revenue). `Recency_Days` correlates negatively with monetary value — customers who
haven't purchased recently also tend to have spent less overall, a classic churn-risk signal.

In [ ]:
sns.pairplot(df[["AnnualIncome_INR_K", "SpendingScore", "PurchaseFrequency", "Recency_Days"]].sample(400, random_state=RANDOM_STATE))
plt.show()

**Insight:** No single pair of features cleanly separates customers into
groups by eye — confirming we need an algorithmic, multi-feature clustering approach rather than
manual rule-based segmentation.

## 6. Feature Engineering — RFM Analysis

RFM (Recency, Frequency, Monetary) is the industry-standard framework for scoring customer value:
- **Recency** — how recently did they buy? (lower = better)
- **Frequency** — how often do they buy? (higher = better)
- **Monetary** — how much do they spend? (higher = better)

We bucket each into quartiles (1=worst, 4=best) and combine them into an `RFM_Score`.

In [ ]:
rfm = build_rfm(df)
rfm.head(10)

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(rfm["RFM_Score"], bins=10, kde=False, color="#55A868")
plt.title("Distribution of Combined RFM Score")
plt.xlabel("RFM Score (3 = lowest value, 12 = highest value)")
plt.show()

**Insight:** RFM scores span the full range, confirming a genuine mix of
high-value and low-value/at-risk customers in the base — a good sign that clustering will find
real, actionable structure rather than one homogeneous blob.

## 7. Encoding & Scaling

K-Means and PCA are distance-based, so every feature must be on the same numeric scale — a
feature like `AnnualIncome_INR_K` (range: hundreds to thousands) would otherwise completely
dominate a feature like `Age` (range: 18-80) purely because of its bigger numbers, not because
it's actually more important.

In [ ]:
df_enc, encoders = encode_categoricals(df, ["Gender", "City", "ProductCategoryPreference"])

cluster_features = [
    "Age", "AnnualIncome_INR_K", "SpendingScore", "PurchaseFrequency",
    "Recency_Days", "Tenure_Months", "MonetaryValue_INR"
]
X_scaled, scaler = scale_features(df_enc, cluster_features)
print("Scaled feature matrix shape:", X_scaled.shape)
pd.DataFrame(X_scaled, columns=cluster_features).describe().round(2)

## 8. Finding the Optimal Number of Clusters

We use two complementary methods:
- **Elbow Method** — plot inertia (within-cluster sum of squares) vs k; look for the "elbow" where adding more clusters stops helping much.
- **Silhouette Score** — measures how well-separated clusters are (range -1 to 1, higher is better).

In [ ]:
opt_df = find_optimal_k(X_scaled, k_range=range(2, 11), random_state=RANDOM_STATE)
opt_df

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(opt_df["k"], opt_df["inertia"], marker="o", color="#4C72B0", label="Inertia (Elbow)")
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("Inertia", color="#4C72B0")
ax2 = ax1.twinx()
ax2.plot(opt_df["k"], opt_df["silhouette"], marker="s", color="#C44E52", label="Silhouette Score")
ax2.set_ylabel("Silhouette Score", color="#C44E52")
plt.title("Elbow Method + Silhouette Score")
fig.tight_layout()
plt.show()

**Insight:** The elbow softens noticeably around **k=4**, and while the pure
silhouette score is technically highest at k=2, a 2-cluster split is too coarse to be useful for
marketing (it would just say "big spenders" vs "everyone else"). **k=4** gives the best balance
of statistical separation and business interpretability — it's the smallest k that still lets us
name distinct, actionable personas. We proceed with **k=4**.

In [ ]:
FINAL_K = 4

## 9. Clustering — K-Means

In [ ]:
km_labels, km_model = run_kmeans(X_scaled, FINAL_K, random_state=RANDOM_STATE)
km_eval = evaluate_clustering(X_scaled, km_labels)
print("K-Means evaluation:", km_eval)

## 10. Clustering — Hierarchical (Agglomerative)

In [ ]:
hc_labels, hc_model = run_hierarchical(X_scaled, FINAL_K, linkage="ward")
hc_eval = evaluate_clustering(X_scaled, hc_labels)
print("Hierarchical evaluation:", hc_eval)

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage as scipy_linkage

Z = scipy_linkage(X_scaled[:200], method="ward")  # subsample for a readable dendrogram
plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="lastp", p=20)
plt.title("Hierarchical Clustering Dendrogram (sample of 200 customers)")
plt.xlabel("Customer clusters"); plt.ylabel("Distance")
plt.show()

## 11. Clustering — DBSCAN

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Use a k-distance plot to pick a sensible eps, then fine-tune with a small grid search
neigh = NearestNeighbors(n_neighbors=10).fit(X_scaled)
dists, _ = neigh.kneighbors(X_scaled)
k_dist = np.sort(dists[:, -1])

plt.figure(figsize=(8, 5))
plt.plot(k_dist)
plt.title("k-Distance Plot (used to choose DBSCAN eps)")
plt.xlabel("Points sorted by distance"); plt.ylabel("10-NN distance")
plt.show()

In [ ]:
best_db = None
for eps_try in [0.8, 0.9, 1.0, 1.1, 1.2, 1.5]:
    for ms_try in [5, 8, 10]:
        labels_try, model_try = run_dbscan(X_scaled, eps=eps_try, min_samples=ms_try)
        eval_try = evaluate_clustering(X_scaled, labels_try)
        if eval_try["n_clusters"] >= 2 and not np.isnan(eval_try["silhouette"]):
            if best_db is None or eval_try["silhouette"] > best_db[2]["silhouette"]:
                best_db = (labels_try, model_try, eval_try, eps_try, ms_try)

db_labels, db_model, db_eval, chosen_eps, chosen_ms = best_db
print(f"Chosen DBSCAN params: eps={chosen_eps}, min_samples={chosen_ms}")
print("DBSCAN evaluation:", db_eval)
print("Cluster sizes:", dict(zip(*np.unique(db_labels, return_counts=True))))

**Insight:** DBSCAN technically reports a higher silhouette score, but look at
the cluster sizes: it finds **one large dense cluster, a large "noise" bucket, and many tiny
1-10 point fragments**. That's a classic sign DBSCAN is not well-suited to this kind of evenly
spread, globular (roughly oval-shaped, not weirdly curved) customer data — density-based
clustering shines on irregular-shaped clusters (like geographic hotspots), not smooth behavioral
gradients like income and spending. For a marketing team that needs a handful of clean, equally
actionable segments, DBSCAN's output isn't usable as-is.

## 12. Algorithm Comparison

In [ ]:
comparison = pd.DataFrame([
    {"Algorithm": "K-Means", **km_eval},
    {"Algorithm": "Hierarchical", **hc_eval},
    {"Algorithm": "DBSCAN", **db_eval},
])
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=comparison, x="Algorithm", y="silhouette", hue="Algorithm", ax=axes[0], legend=False, palette="Blues_d")
axes[0].set_title("Silhouette Score (higher = better)")
sns.barplot(data=comparison, x="Algorithm", y="davies_bouldin", hue="Algorithm", ax=axes[1], legend=False, palette="Oranges_d")
axes[1].set_title("Davies-Bouldin Index (lower = better)")
plt.tight_layout()
plt.show()

**Verdict:** We select **K-Means (k=4)** as our production model. It gives
well-balanced, equally-sized, business-interpretable segments; Hierarchical clustering gives
nearly identical results (validating K-Means' structure is real, not an artifact) and its
dendrogram is a nice visual aid; DBSCAN's numeric edge doesn't translate into a usable
segmentation for a marketing team. **K-Means is the model we ship.**

## 13. Dimensionality Reduction (PCA) & Visualization

In [ ]:
X_pca, pca_model = reduce_dimensions(X_scaled, n_components=2, random_state=RANDOM_STATE)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=km_labels, cmap="viridis", alpha=0.7)
plt.xlabel(f"PC1 ({pca_model.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca_model.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title(f"Customer Segments (K-Means, k={FINAL_K}) — PCA 2D Projection")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

X_pca3, pca_model3 = reduce_dimensions(X_scaled, n_components=3, random_state=RANDOM_STATE)
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
p = ax.scatter(X_pca3[:, 0], X_pca3[:, 1], X_pca3[:, 2], c=km_labels, cmap="viridis", alpha=0.7)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
ax.set_title(f"Customer Segments — PCA 3D Projection (k={FINAL_K})")
fig.colorbar(p, label="Cluster", shrink=0.6)
plt.tight_layout()
plt.show()

**Insight:** Even after compressing 7 features down to 2-3 principal
components, the four clusters remain visually distinguishable — strong evidence the segments
reflect genuine structure in customer behavior, not an artifact of the algorithm.

## 14. Cluster Profiling & Naming

Now we translate cluster numbers into personas a marketing team can actually use.

In [ ]:
df_profile = df.copy()
df_profile["Cluster"] = km_labels

profile = df_profile.groupby("Cluster")[cluster_features].mean().round(2)
profile["CustomerCount"] = df_profile.groupby("Cluster").size()
profile["PctOfBase"] = (profile["CustomerCount"] / len(df_profile) * 100).round(1)
profile

In [ ]:
SEGMENT_NAMES = {
    2: "Premium Spenders (High-Value Loyal Customers)",
    0: "Steady Mature Regulars",
    1: "New Bargain Hunters",
    3: "At-Risk Customers",
}
profile["SegmentName"] = profile.index.map(SEGMENT_NAMES)
profile[["SegmentName"] + cluster_features + ["CustomerCount", "PctOfBase"]]

**How we named them (based on the table above):**

| Cluster | Name | Why |
|---|---|---|
| 2 | **Premium Spenders** | Highest income, highest spending score, most frequent buyers, most recent purchases, longest tenure, highest revenue per customer. The company's best customers. |
| 0 | **Steady Mature Regulars** | Older average age, mid income/spending, moderate recency — dependable, consistent, unexciting-but-valuable customers. |
| 1 | **New Bargain Hunters** | Younger, lower-mid income, decent spending score and frequency, good recency — price-sensitive but engaged and growing. |
| 3 | **At-Risk Customers** | Relatively high income but the **lowest spending score, lowest frequency, and by far the highest recency (~120 days since last purchase)** — customers who *can* spend but have quietly stopped engaging. Highest churn risk. |


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, col in zip(axes.flat, ["AnnualIncome_INR_K", "SpendingScore", "PurchaseFrequency", "Recency_Days"]):
    sns.barplot(x=profile.index, y=profile[col], hue=profile.index, ax=ax, legend=False, palette="viridis")
    ax.set_title(f"Avg {col} by Cluster")
    ax.set_xlabel("Cluster")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
sizes = df_profile["Cluster"].value_counts().sort_index()
labels_pie = [f"{SEGMENT_NAMES[i]}\n({sizes[i]} customers)" for i in sizes.index]
plt.pie(sizes, labels=labels_pie, autopct="%1.1f%%", colors=sns.color_palette("viridis", len(sizes)))
plt.title("Customer Segment Distribution")
plt.tight_layout()
plt.show()

## 15. Business Recommendations by Segment

**🏆 Premium Spenders (21% of base, ~₹68.8L combined avg. monetary value)**
- Enroll in a VIP/early-access loyalty tier; they respond to exclusivity, not discounts.
- Assign priority customer support and personal shopper-style recommendations.
- Use for referral programs — their word-of-mouth carries weight.

**🧭 Steady Mature Regulars (24% of base)**
- Cross-sell complementary products via targeted, non-discount email campaigns.
- Introduce a mid-tier loyalty program to nudge them toward "Premium" behavior.
- Survey for satisfaction — small friction fixes could unlock more frequency.

**🛍️ New Bargain Hunters (28% of base — the largest segment)**
- Time-boxed discount codes and bundle deals convert this price-sensitive group well.
- Gamify early engagement (referral bonuses, first-3-orders rewards) to build habit and tenure.
- This is the segment with the most *growth* upside — nurture them into Premium Spenders over time.

**🚨 At-Risk Customers (27% of base)**
- Immediate win-back campaign: personalized "we miss you" email + a meaningful (not token) incentive.
- Investigate root cause via a short feedback survey — since income is high, price isn't the barrier.
- If no response after win-back, deprioritize marketing spend on this group to protect ROI.


## 16. Model Evaluation Summary

In [ ]:
print("=== Final Model: K-Means (k=4) ===")
print(f"Silhouette Score:      {km_eval['silhouette']:.4f}  (range -1 to 1, higher is better)")
print(f"Davies-Bouldin Index:  {km_eval['davies_bouldin']:.4f}  (lower is better)")

## 17. Cluster Stability Check

A good clustering result shouldn't change wildly if we re-run it on a slightly different sample
of customers or a different random seed. We re-run K-Means 10 times on random 80% subsamples and
check how much the silhouette score varies.

In [ ]:
stability = cluster_stability(X_scaled, FINAL_K, n_runs=10, random_state=RANDOM_STATE)
print(stability)

**Insight:** A very small standard deviation relative to the mean silhouette
score tells us the 4-cluster structure is **stable** — it's not an artifact of one particular
random seed or sample, so marketing can trust these segments to hold up on new customer data
too.

## 18. Export Final Labeled Dataset

Save the customer-level cluster assignments so they can be pushed into a CRM, BI dashboard
(Streamlit app in `app.py`), or marketing automation tool.

In [ ]:
df_profile["SegmentName"] = df_profile["Cluster"].map(SEGMENT_NAMES)
df_profile.to_csv("../data/customer_data_with_segments.csv", index=False)
profile.to_csv("../data/cluster_profile.csv")
print("Saved: data/customer_data_with_segments.csv, data/cluster_profile.csv")
df_profile[["CustomerID", "Cluster", "SegmentName"]].head(10)

## 19. Conclusion

Using K-Means clustering (k=4, validated against Hierarchical and DBSCAN, PCA-visualized, and
stability-checked), we split the customer base into **four actionable segments** — Premium
Spenders, Steady Mature Regulars, New Bargain Hunters, and At-Risk Customers — each with a
distinct behavioral fingerprint and a tailored marketing strategy.

**Next steps for the business:** pilot the win-back campaign on At-Risk Customers first (fastest
path to recovering lost revenue), then build the VIP tier for Premium Spenders, and track segment
migration month-over-month to see whether New Bargain Hunters are graduating into higher-value
tiers.

See `reports/executive_summary.pdf` for a business-facing summary of these findings, and `app.py`
for the interactive Streamlit dashboard.
